In [ ]:
import pandas as pd
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition, Element
from tqdm import tqdm

In [ ]:
nanozyme = pd.read_csv('./data/catalase_features.csv')

In [ ]:
nanozyme['label'] = 1

In [ ]:
mp = pd.read_csv('../data/materials_project_features.csv')

In [ ]:
mp['label'] = 0

In [ ]:
data = pd.concat((nanozyme, mp), axis=0)

In [ ]:
data = data.reset_index(drop=True)

In [ ]:
data['pearson'] = data['pearson'].str.lower()

In [ ]:
exclude_cols = ['formula', 'space', 'pearson']
cols_to_fill = [col for col in data.columns if col not in exclude_cols]
data[cols_to_fill] = data[cols_to_fill].fillna(data[cols_to_fill].median())

In [ ]:
for col in ['space', 'pearson']:
    if col in data.columns:
        mode_val = data[col].mode().iloc[0]
        data[col] = data[col].fillna(mode_val)

In [ ]:
ep_feat = ElementProperty.from_preset("magpie")
feature_names = ep_feat.feature_labels()

magpie_features = []
for formula in tqdm(data['formula']):
    comp = Composition(formula)
    features = ep_feat.featurize(comp)
    magpie_features.append(features)

In [ ]:
magpie_df = pd.DataFrame(magpie_features, columns=feature_names)
data_v2 = pd.concat([data.reset_index(drop=True), magpie_df], axis=1)

In [ ]:
def has_transition_metal(formula: str) -> bool:
    comp = Composition(formula)
    for el in comp.elements:
        if Element(el.symbol).is_transition_metal:
            return True
    return False

data_v2['has_tm'] = data_v2['formula'].apply(has_transition_metal)

In [ ]:
data_v2.to_csv('./data/catalase_data.csv', index=False)